In [16]:

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from typing import Literal

class llm_schema(BaseModel):
    movie_summery_flag: Literal["positive", "negative"]


promot_templet = ChatPromptTemplate.from_messages([
    ("system","You movie review"),
    ("human", "Plz summerize the movie in brief {input}")
])



## Task 2

llm_openai=ChatOpenAI(model='gpt-5-mini',temperature=0)

llm_strctured_output=llm_openai.with_structured_output(llm_schema)

## Task: 3  Str output parser .content
str_parser = StrOutputParser()








In [17]:

from langchain_core.runnables import RunnableLambda
## Task 4:- create a dic maker
def pydantic_json(input: llm_schema) -> str:
    return input.model_dump()['movie_summery_flag']



## Conver that fun to runnable lambda
pydantic_json_runnable = RunnableLambda(pydantic_json)


## CHAIN With Condition 


## Paraller chain 1 ##

In [ ]:

## Task 1 ##

linkdin_promot = ChatPromptTemplate.from_messages([
    ("system","your are a Linkdin post genrater"),
    ('human',"Create a post for follwing text for linkedin : {text}")
])


llm_openai = ChatOpenAI(model='gpt-5-mini',temperature=0)

str_parser= StrOutputParser()

chian_linkedin = linkdin_promot | llm_openai | str_parser

In [1]:
## Paraller chain 2 ##


In [3]:
def insta_chain(text:str):

    text=text['text']
    insta_promot = ChatPromptTemplate.from_messages([
        ("system","your are a instagram post genrater"),
        ('human',"Create a post for follwing text for instagram : {text}")
    ])

    llm_openai=ChatOpenAI(model='gpt-5-mini',temperature=0)

    str_parser= StrOutputParser()   

    chain_insta = insta_promot | llm_openai | str_parser

    result =chain_insta.invoke(text)

    return result


insta_chain_runnable = RunnableLambda(insta_chain)

## Final Orchastration ##

In [20]:
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnableBranch

conditional_chain= RunnableBranch (
    (lambda x : 'positive' in x , chian_linkedin),
    insta_chain_runnable
)


final_result_ans= promot_templet | llm_strctured_output | pydantic_json_runnable |conditional_chain

final_result_ans.invoke({'input':"i love this KGF movie"})

'Positivity is a choice — and it changes everything. ✨\n\nThis week I watched a team turn a tight deadline into an opportunity: instead of focusing on what could go wrong, they focused on what we could do next. The result? Faster decisions, better collaboration, and a real sense of momentum.\n\nA few simple habits that helped:\n- Name small wins every day\n- Shift from “problem” to “next step”\n- Thank someone for their contribution\n\nPositivity doesn’t ignore challenges — it helps us face them with clarity and resilience. How do you keep a positive mindset at work? Share one practice that helps you stay focused and upbeat. 🙌\n\n#Positivity #Leadership #Teamwork #Growth #Wellbeing'

In [ ]:

final_result = promot_templet | llm_openai | pydantic_json_runnable | conditional_chain

final_result.invoke({"input":"i love KFG Moive"})